# Hackology II — Profile predict_v2 на T4

Цель: измерить РЕАЛЬНОЕ время инференса на T4 (16GB / 30min budget — та же среда что финал) для разных конфигов, выбрать самый тяжёлый который влезает.

## Лестница (сверху вниз — от тяжёлого к лёгкому)

| Level | Команда | Ожидаемое время (481 imgs) | mAP |
|---|---|---|---|
| L4 | `--mode heavy --models student teacher` | ~40-60 мин ⚠️ | топ |
| L3 | `--mode balanced --models student teacher` | ~20-30 мин | очень хорошо |
| L2 | `--mode heavy --model student` | ~12-18 мин ✅ | хорошо |
| L1 | `--mode balanced --model student` | ~6-10 мин ✅ | бейзлайн |

**Бюджет финала: 30 мин**. Берём САМОЕ ТЯЖЁЛОЕ что влезает в ~25 мин с запасом.

## Перед запуском
1. Runtime → Change runtime type → **T4 GPU**
2. (опц) для последнего шага `git push` — добавь `GITHUB_PAT` в Colab Secrets (иконка 🔑). Для самого профайла не нужен — репо публичный.

## 0. GPU check

In [ ]:
!nvidia-smi

## 1. Install uv

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
!uv --version

## 2. Клонируем репо (публичный — без аутентификации)

In [ ]:
import os

REPO_URL = "https://github.com/qwontie/hackology2.git"
REPO_DIR = "/content/hackology2"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
os.chdir(REPO_DIR)
!pwd

## 3. `uv sync` — устанавливает закреплённый env (torch 2.6.0+cu124, ultralytics 8.3.24, ...)

На свежем Colab займёт ~2 мин. Качает с правильного cu124 индекса (это критично — без этого `ncclCommResume` сломает torch).

In [ ]:
!uv sync --frozen

In [ ]:
# Sanity: torch видит T4
!uv run python -c "import torch; print(f'torch={torch.__version__} cuda={torch.cuda.is_available()} dev={torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}')"

## 4. Скачать датасет

Качает весь датасет (4.3GB → ~3-5 мин на Colab). Нам реально нужен только `data/public_test/images/` (467MB, 481 img) — но `download_data.sh` тянет всё.

In [ ]:
!bash download_data.sh

In [ ]:
# Проверь что public_test приехал
import os
n = len([f for f in os.listdir('data/public_test/images') if f.endswith(('.jpg', '.jpeg', '.png'))])
print(f'public_test images: {n}')
assert n == 481, f'Expected 481 imgs, got {n}'

## 5. Warmup — скачивает веса с GH Release, прогревает CUDA кернели

Первый запуск медленнее из-за: (a) скачка `student.pt` (~40MB) с GH, (b) JIT компиляция CUDA операций. Делаем на 20 картинках чтобы быстро.

In [ ]:
# Закинем 20 первых картинок в отдельную папку для warmup
!mkdir -p /tmp/warmup && cp $(ls data/public_test/images/*.jpg | head -20) /tmp/warmup/
!ls /tmp/warmup | wc -l

!time uv run python predict_v2.py \
  --input /tmp/warmup \
  --annotations data/public_test/test_images.json \
  --output /tmp/warmup.json \
  --mode balanced --model student

## 6. L1 — student balanced (single 1536 + flip)

**Это БЕЙЗЛАЙН — гарантированно влезает.** Все остальные уровни должны быть лучше иначе берём это.

In [ ]:
!time uv run python predict_v2.py \
  --input data/public_test/images \
  --annotations data/public_test/test_images.json \
  --output /tmp/L1_student_balanced.json \
  --mode balanced --model student

## 7. L2 — student heavy (multi-scale [1280,1536,1920] + flip + WBF)

Single model, 3 шкалы. Если ≤25 мин — берём над L1.

In [ ]:
!time uv run python predict_v2.py \
  --input data/public_test/images \
  --annotations data/public_test/test_images.json \
  --output /tmp/L2_student_heavy.json \
  --mode heavy --model student

## 8. L3 — student + teacher balanced (2-model WBF ensemble, 1536 + flip)

Тут teacher.pt подкачается (~110MB).

In [ ]:
!time uv run python predict_v2.py \
  --input data/public_test/images \
  --annotations data/public_test/test_images.json \
  --output /tmp/L3_ensemble_balanced.json \
  --mode balanced --models student teacher

## 9. L4 — student + teacher heavy (топ-конфиг, скорее всего НЕ влезет)

2 модели × 3 шкалы × flip = 12 проходов на картинку. Может занять 40+ мин. **Можно прервать (■) если видно что не влезет в 25 мин.**

In [ ]:
!time uv run python predict_v2.py \
  --input data/public_test/images \
  --annotations data/public_test/test_images.json \
  --output /tmp/L4_ensemble_heavy.json \
  --mode heavy --models student teacher

## 10. Сводка таймингов + sanity check

In [ ]:
import json, os
from pathlib import Path

for level, path in [
    ('L1 student balanced', '/tmp/L1_student_balanced.json'),
    ('L2 student heavy',    '/tmp/L2_student_heavy.json'),
    ('L3 ensemble balanced', '/tmp/L3_ensemble_balanced.json'),
    ('L4 ensemble heavy',   '/tmp/L4_ensemble_heavy.json'),
]:
    if not os.path.exists(path):
        print(f'{level:30s} — SKIPPED')
        continue
    preds = json.loads(Path(path).read_text())
    n_imgs = len(set(p['image_id'] for p in preds))
    n_cats = len(set(p['category_id'] for p in preds))
    size_mb = os.path.getsize(path) / 1024**2
    print(f'{level:30s} preds={len(preds):>6}  imgs={n_imgs:>3}  cats={n_cats:>3}  size={size_mb:.1f}MB')

## 11. Submit чемпиона на public leaderboard

Поменяй `WINNER` на путь до самого тяжёлого варианта который ВЛЕЗ В БЮДЖЕТ. Чтобы пушнуть — нужен `GITHUB_PAT` в Colab Secrets (иконка 🔑 → новый secret → права repo).

In [ ]:
WINNER = '/tmp/L3_ensemble_balanced.json'  # ← поменяй на тот что выбираем

import shutil, os
shutil.copy(WINNER, 'submissions/predictions.json')
print('Copied:', WINNER, '->', 'submissions/predictions.json')

# Получаем PAT для push'а
try:
    from google.colab import userdata
    PAT = userdata.get('GITHUB_PAT')
except Exception:
    PAT = input('GitHub PAT (для push): ')

!git config user.name  "Colab"
!git config user.email "team@hackology.dev"
!git remote set-url origin https://{PAT}@github.com/qwontie/hackology2.git
!git add submissions/predictions.json
!git commit -m "submission: profiled ensemble"
!git push